In [1]:
import pandas as pd

In [11]:
data = pd.read_csv("clinical_analytics.csv")
print(data.shape)
data.head()

(81667, 13)


,Admit Source,Admit Type,Appt Start Time,Care Score,Check-In Time,Clinic Name,Department,Diagnosis Primary,Discharge Datetime new,Encounter Number,Encounter Status,Number of Records,Wait Time Min
0,Emergency Room,Emergency,2014-01-02 11:38:50 PM,2,2014-01-02 11:24:00 PM,Madison Center,Cardiology,NaN,NaN,P7P4KC587,Cancelled,1,14
1,Emergency Room,Emergency,2014-01-02 11:44:18 PM,2,2014-01-02 11:24:00 PM,Madison Center,Cardiology,NaN,NaN,P7P4KC587,Cancelled,1,20
2,Emergency Room,Emergency,2014-01-02 11:47:11 PM,2,2014-01-02 11:24:00 PM,Madison Center,Cardiology,NaN,NaN,P7P4KC587,Cancelled,1,23
3,Emergency Room,Emergency,2014-01-08 10:38:04 PM,4,2014-01-08 9:00:00 PM,Madison Center,Emergency,NaN,NaN,PK559C587,Cancelled,1,98
4,Emergency Room,Emergency,2014-01-09 12:00:26 AM,4,2014-01-08 10:47:00 PM,Madison Center,Emergency,NaN,NaN,48559C587,Cancelled,1,73


In [3]:
data["Admit Source"].unique()

array(['Emergency Room', 'Clinic Referral', 'Physician Referral', nan,
       'Outside Health Care Facility',
       'Admitted as transfer from another unit', 'Court/Law Enforcement',
       'Outside Hospital', 'Transfer from Critical Access Hospital',
       'Outside Home Health Agency', 'Information Unavailable',
       'Psych, Substance Abuse, or Rehab Hospital'], dtype=object)

In [4]:
data["Encounter Status"].unique()

array(['Cancelled', 'Admit', 'Preadmit', 'ED Discharged', 'Discharged',
       'Inpatient Discharged', 'OBS Discharged', 'Day Surg Discharged',
       nan], dtype=object)

In [5]:
data["Number of Records"].unique()

array([1])

In [6]:
data["Department"].unique()

array(['Cardiology', 'Emergency', 'Ophthalmology', 'Oral Surgery',
       'General Surgery', 'Neurosurgery', 'Urology', 'Otolaryngology',
       'Gastroenterology', 'Plastic Surgery', 'Neurology',
       'General Medicine', 'Rehabilitation', 'Nephrology',
       'Dental Surgery', 'Orthopedics', 'Cardiac Surgery'], dtype=object)

In [14]:
pip install dash-bootstrap-components

Note: you may need to restart the kernel to use updated packages.


In [9]:
# ================================
# 📦 IMPORTACIONES
# ================================
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc

In [15]:
# Tema visual
px.defaults.template = "simple_white"

# ================================
# 📥 CARGA DE DATOS
# ================================
df = pd.read_csv("clinical_analytics.csv")

# Limpiar nombres de columnas
df.columns = df.columns.str.strip()

# Convertir fecha
df['Appt Start Time'] = pd.to_datetime(df['Appt Start Time'], errors='coerce')

# Eliminar valores nulos en fecha
df = df.dropna(subset=['Appt Start Time'])

# ================================
# 🚀 INICIALIZAR APP
# ================================
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# ================================
# 🎨 DISEÑO (LAYOUT)
# ================================
app.layout = dbc.Container([

    # 🔹 TÍTULO
    dbc.Row([
        dbc.Col(html.H2("🏥 Dashboard Operativo - Clínica", className="text-center mb-4"))
    ]),

    # 🔹 FILTROS
    dbc.Row([
        dbc.Col([
            dcc.DatePickerRange(
                id='filtro_fecha',
                start_date=df['Appt Start Time'].min(),
                end_date=df['Appt Start Time'].max()
            )
        ], width=4),

        dbc.Col([
            dcc.Dropdown(
                id='filtro_clinica',
                options=[{'label': c, 'value': c} for c in df['Clinic Name'].dropna().unique()],
                multi=True,
                placeholder="Selecciona clínicas"
            )
        ], width=4),

        dbc.Col([
            dcc.Dropdown(
                id='filtro_fuente',
                options=[{'label': s, 'value': s} for s in df['Admit Source'].dropna().unique()],
                multi=True,
                placeholder="Fuente de admisión"
            )
        ], width=4),
    ]),

    html.Br(),

    # 🔹 KPIs
    dbc.Row([
        dbc.Col(html.Div(id="kpi_pacientes"), width=4),
        dbc.Col(html.Div(id="kpi_espera"), width=4),
        dbc.Col(html.Div(id="kpi_satisfaccion"), width=4),
    ]),

    html.Br(),

    # 🔹 GRÁFICOS
    dbc.Row([
        dbc.Col(dcc.Graph(id="grafico_volumen"), width=4),
        dbc.Col(dcc.Graph(id="grafico_espera"), width=4),
        dbc.Col(dcc.Graph(id="grafico_satisfaccion"), width=4),
    ])

], fluid=True)

# ================================
# 🔄 CALLBACK
# ================================
@app.callback(
    [
        Output("grafico_volumen", "figure"),
        Output("grafico_espera", "figure"),
        Output("grafico_satisfaccion", "figure"),
        Output("kpi_pacientes", "children"),
        Output("kpi_espera", "children"),
        Output("kpi_satisfaccion", "children"),
    ],
    [
        Input("filtro_fecha", "start_date"),
        Input("filtro_fecha", "end_date"),
        Input("filtro_clinica", "value"),
        Input("filtro_fuente", "value"),
    ]
)
def actualizar_dashboard(fecha_inicio, fecha_fin, clinicas, fuentes):

    datos = df.copy()

    # 🔹 FILTRO POR FECHA
    if fecha_inicio and fecha_fin:
        datos = datos[
            (datos['Appt Start Time'] >= fecha_inicio) &
            (datos['Appt Start Time'] <= fecha_fin)
        ]

    # 🔹 FILTROS
    if clinicas:
        datos = datos[datos['Clinic Name'].isin(clinicas)]

    if fuentes:
        datos = datos[datos['Admit Source'].isin(fuentes)]

    # ================================
    # 📊 GRÁFICOS
    # ================================

    # 🔹 VOLUMEN DE PACIENTES
    volumen = datos.groupby("Clinic Name").size().reset_index(name="Pacientes")
    volumen = volumen.sort_values(by="Pacientes", ascending=False)

    fig_volumen = px.bar(
        volumen,
        x="Clinic Name",
        y="Pacientes",
        text="Pacientes",
        color="Pacientes",
        color_continuous_scale="Blues",
        title="📊 Volumen de Pacientes por Clínica"
    )

    fig_volumen.update_traces(textposition='outside')
    fig_volumen.update_layout(
        xaxis_title="Clínica",
        yaxis_title="Cantidad",
        title_x=0.5
    )

    # 🔹 TIEMPO DE ESPERA
    fig_espera = px.box(
        datos,
        x="Department",
        y="Wait Time Min",
        color="Department",
        title="⏳ Tiempo de Espera por Departamento"
    )

    promedio_espera = datos['Wait Time Min'].mean()

    fig_espera.add_hline(
        y=promedio_espera,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Promedio: {round(promedio_espera,2)} min"
    )

    fig_espera.update_layout(
        xaxis_title="Departamento",
        yaxis_title="Minutos",
        showlegend=False,
        title_x=0.5
    )

    # 🔹 SATISFACCIÓN
    fig_satisfaccion = px.histogram(
        datos,
        x="Care Score",
        nbins=20,
        color="Department",
        opacity=0.7,
        barmode="overlay",
        title="⭐ Distribución de Satisfacción"
    )

    fig_satisfaccion.update_layout(
        xaxis_title="Calificación",
        yaxis_title="Frecuencia",
        title_x=0.5
    )

    # ================================
    # 📌 KPIs
    # ================================
    total_pacientes = len(datos)
    tiempo_promedio = round(datos['Wait Time Min'].mean(), 2)
    satisfaccion_promedio = round(datos['Care Score'].mean(), 2)

    kpi_pacientes = dbc.Card(dbc.CardBody([
        html.H6("👥 Total Pacientes"),
        html.H3(total_pacientes)
    ]), color="primary", inverse=True)

    kpi_espera = dbc.Card(dbc.CardBody([
        html.H6("⏳ Tiempo Promedio"),
        html.H3(f"{tiempo_promedio} min")
    ]), color="warning", inverse=True)

    kpi_satisfaccion = dbc.Card(dbc.CardBody([
        html.H6("⭐ Satisfacción"),
        html.H3(satisfaccion_promedio)
    ]), color="success", inverse=True)

    return fig_volumen, fig_espera, fig_satisfaccion, kpi_pacientes, kpi_espera, kpi_satisfaccion


# ================================
# ▶️ EJECUCIÓN
# ================================
if __name__ == "__main__":
    app.run(debug=True, port=8051) 

C:\Users\mayk_\AppData\Local\Temp\ipykernel_40292\3361578442.py:13: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



In [16]:
# ================================
# 📦 IMPORTACIONES
# ================================
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc

# Tema gráfico
px.defaults.template = "simple_white"

# ================================
# 📥 CARGA DE DATOS
# ================================
df = pd.read_csv("clinical_analytics.csv")

df.columns = df.columns.str.strip()

df['Appt Start Time'] = pd.to_datetime(df['Appt Start Time'], errors='coerce')

df = df.dropna(subset=['Appt Start Time'])

# ================================
# 🚀 APP + TEMA
# ================================
app = Dash(__name__, external_stylesheets=[dbc.themes.CYBORG])

# ================================
# 🎨 ESTILOS
# ================================
CARD_STYLE = {
    "borderRadius": "15px",
    "boxShadow": "0px 4px 10px rgba(0,0,0,0.4)",
    "padding": "10px"
}

# ================================
# 🎨 LAYOUT
# ================================
app.layout = dbc.Container([

    # 🔷 NAVBAR
    dbc.Navbar(
        dbc.Container([
            dbc.NavbarBrand("🏥 Dashboard Clínico", className="fw-bold fs-4"),
        ]),
        color="dark",
        dark=True,
        className="mb-4"
    ),

    # 🔷 FILTROS
    dbc.Card([
        dbc.CardBody([
            dbc.Row([

                dbc.Col([
                    html.Label("📅 Rango de fechas"),
                    dcc.DatePickerRange(
                        id='filtro_fecha',
                        start_date=df['Appt Start Time'].min(),
                        end_date=df['Appt Start Time'].max()
                    )
                ], md=4),

                dbc.Col([
                    html.Label("🏥 Clínica"),
                    dcc.Dropdown(
                        id='filtro_clinica',
                        options=[{'label': c, 'value': c} for c in df['Clinic Name'].dropna().unique()],
                        multi=True
                    )
                ], md=4),

                dbc.Col([
                    html.Label("🚑 Fuente"),
                    dcc.Dropdown(
                        id='filtro_fuente',
                        options=[{'label': s, 'value': s} for s in df['Admit Source'].dropna().unique()],
                        multi=True
                    )
                ], md=4),

            ])
        ])
    ], style=CARD_STYLE, className="mb-4"),

    # 🔷 KPIs
    dbc.Row([
        dbc.Col(html.Div(id="kpi_pacientes"), md=4),
        dbc.Col(html.Div(id="kpi_espera"), md=4),
        dbc.Col(html.Div(id="kpi_satisfaccion"), md=4),
    ], className="mb-4"),

    # 🔷 GRÁFICOS
    dbc.Row([

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id="grafico_volumen")
                ])
            ], style=CARD_STYLE),
            md=4
        ),

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id="grafico_espera")
                ])
            ], style=CARD_STYLE),
            md=4
        ),

        dbc.Col(
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id="grafico_satisfaccion")
                ])
            ], style=CARD_STYLE),
            md=4
        ),

    ])

], fluid=True)

# ================================
# 🔄 CALLBACK
# ================================
@app.callback(
    [
        Output("grafico_volumen", "figure"),
        Output("grafico_espera", "figure"),
        Output("grafico_satisfaccion", "figure"),
        Output("kpi_pacientes", "children"),
        Output("kpi_espera", "children"),
        Output("kpi_satisfaccion", "children"),
    ],
    [
        Input("filtro_fecha", "start_date"),
        Input("filtro_fecha", "end_date"),
        Input("filtro_clinica", "value"),
        Input("filtro_fuente", "value"),
    ]
)
def actualizar_dashboard(fecha_inicio, fecha_fin, clinicas, fuentes):

    datos = df.copy()

    # 🔹 FILTROS
    if fecha_inicio and fecha_fin:
        datos = datos[
            (datos['Appt Start Time'] >= fecha_inicio) &
            (datos['Appt Start Time'] <= fecha_fin)
        ]

    if clinicas:
        datos = datos[datos['Clinic Name'].isin(clinicas)]

    if fuentes:
        datos = datos[datos['Admit Source'].isin(fuentes)]

    # ================================
    # 📊 GRÁFICOS
    # ================================

    # 🔹 VOLUMEN
    volumen = datos.groupby("Clinic Name").size().reset_index(name="Pacientes")
    volumen = volumen.sort_values(by="Pacientes", ascending=False)

    fig_volumen = px.bar(
        volumen,
        x="Clinic Name",
        y="Pacientes",
        text="Pacientes",
        color="Pacientes",
        color_continuous_scale="Blues",
        title="📊 Volumen de Pacientes"
    )

    fig_volumen.update_traces(textposition='outside')
    fig_volumen.update_layout(title_x=0.5)

    # 🔹 ESPERA
    fig_espera = px.box(
        datos,
        x="Department",
        y="Wait Time Min",
        color="Department",
        title="⏳ Tiempo de Espera"
    )

    promedio = datos['Wait Time Min'].mean()

    fig_espera.add_hline(
        y=promedio,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Promedio: {round(promedio,2)}"
    )

    fig_espera.update_layout(title_x=0.5, showlegend=False)

    # 🔹 SATISFACCIÓN
    fig_satisfaccion = px.histogram(
        datos,
        x="Care Score",
        nbins=20,
        color="Department",
        opacity=0.7,
        barmode="overlay",
        title="⭐ Satisfacción"
    )

    fig_satisfaccion.update_layout(title_x=0.5)

    # ================================
    # 📌 KPIs
    # ================================
    total = len(datos)
    avg_wait = round(datos['Wait Time Min'].mean(), 2)
    avg_score = round(datos['Care Score'].mean(), 2)

    kpi_pacientes = dbc.Card(
        dbc.CardBody([
            html.H6("👥 Pacientes"),
            html.H2(total)
        ]),
        color="primary",
        inverse=True,
        style=CARD_STYLE
    )

    kpi_espera = dbc.Card(
        dbc.CardBody([
            html.H6("⏳ Espera Promedio"),
            html.H2(f"{avg_wait} min")
        ]),
        color="warning",
        inverse=True,
        style=CARD_STYLE
    )

    kpi_satisfaccion = dbc.Card(
        dbc.CardBody([
            html.H6("⭐ Satisfacción"),
            html.H2(avg_score)
        ]),
        color="success",
        inverse=True,
        style=CARD_STYLE
    )

    return fig_volumen, fig_espera, fig_satisfaccion, kpi_pacientes, kpi_espera, kpi_satisfaccion


# ================================
# ▶️ RUN
# ================================
if __name__ == "__main__":
    app.run(debug=True, port=8051)

C:\Users\mayk_\AppData\Local\Temp\ipykernel_40292\2667246131.py:19: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

